### Retrieval

**RAG의 5단계**
1. **Document Loader**: 문서를 불러오고
2. **Document Transformer**: 문서를 쪼개고
3. **Embedding**: 텍스트를 숫자로 바꾸고
4. **Vector Store**: 저장소에 넣고
5. **Retrieval**: 검색해서 LLM에 전달합니다.

In [1]:
%pip install langchain-community pypdf faiss-cpu sentence-transformers

  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached pypdf-6.5.0-py3-none-any.whl.metadata (7.1 kB)
  Using cached faiss_cpu-1.13.1-cp312-cp312-win_amd64.whl.metadata (7.6 kB)
  Using cached sentence_transformers-5.2.0-py3-none-any.whl.metadata (16 kB)
  Using cached aiohttp-3.13.2-cp312-cp312-win_amd64.whl.metadata (8.4 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached pydantic_settings-2.12.0-py3-none-any.whl.metadata (3.4 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlist-1.8.0-cp312-cp312-win_amd64.whl.metadata (21 kB)
  Using cached multidict-6.7.0-cp312-cp312-win_amd64.whl.metadata (5.5 kB)
  Using cached propcache-0.4.1-cp312-cp312-win_amd64.whl.metadata (14 kB)
  Using cached yarl-1.22.0-cp312-cp312-win_amd64.whl.metada

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

### Document Loader(문서 불러오기)

In [4]:
%pip install bs4


   ---------------------------------------- 3/3 [bs4]

Note: you may need to restart the kernel to use updated packages.


In [12]:
from langchain_community.document_loaders import WebBaseLoader # 웹페이지 URL에서 텍스트를 긁어오는 도구

url = "https://ko.wikipedia.org/wiki/%EC%9C%84%ED%82%A4%EB%B0%B1%EA%B3%BC:%EC%A0%95%EC%B1%85%EA%B3%BC_%EC%A7%80%EC%B9%A8"

# 로더 인스턴스 생성
loader = WebBaseLoader(url)

# 해당 URL에 접속하여 HTML 파싱, 텍스트만 추출하여 Document 객체 리스트로 반환.
documents = loader.load()

print(len(documents))
print(documents[0].metadata)

# 본문 내용 확인
print(documents[0].page_content[:500])

1
{'source': 'https://ko.wikipedia.org/wiki/%EC%9C%84%ED%82%A4%EB%B0%B1%EA%B3%BC:%EC%A0%95%EC%B1%85%EA%B3%BC_%EC%A7%80%EC%B9%A8', 'title': '위키백과:정책과 지침 - 위키백과, 우리 모두의 백과사전', 'language': 'ko'}




위키백과:정책과 지침 - 위키백과, 우리 모두의 백과사전



























본문으로 이동







주 메뉴





주 메뉴
사이드바로 이동
숨기기



		둘러보기
	


대문최근 바뀜요즘 화제임의의 문서로





		사용자 모임
	


사랑방사용자 모임관리 요청





		편집 안내
	


소개도움말정책과 지침질문방



















검색











검색






















보이기
















기부

계정 만들기

로그인








개인 도구





기부 계정 만들기 로그인




























목차
사이드바로 이동
숨기기




처음 위치





1
최상위 정책








2
'정책과 지침'이란?








3
준수








4
집행








5
문서 내용








6
정책과 지침은 백과사전의 일부가 아닙니다






In [ ]:
from langchain_community.document_loaders import PyPDFLoader # PDF 파일을 로드하여 텍스트로 변환하는 도구

# 로더 인스턴스 생성 (파일 경로 지정)
loader = PyPDFLoader("The_Adventures_of_Tom_Sawyer.pdf")

# 문서 로드 실행 : PDF 각 페이지를 하나의 Document 객체로 변환하여 리스트로 반환
documents = loader.load()

print(len(documents))
print(documents[0].metadata)
print(documents[3].page_content)    # 4번째 페이지(인덱스 3)의 본문 내용

35
{'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'creator': 'Acrobat PDFMaker 7.0 dla programu Word', 'creationdate': '2006-08-26T00:50:00+02:00', 'author': 'GOLDEN', 'company': 'c', 'title': 'Microsoft Word - 1', 'moddate': '2021-01-27T15:00:11+01:00', 'source': 'The_Adventures_of_Tom_Sawyer.pdf', 'total_pages': 35, 'page': 0, 'page_label': '1'}
Pearson Education Limited                                                                            
Edinburgh Gate, Harlow,                                                                               
Essex CM20 2JE, England                                                                              
and Associated Companies throughout the world. 
ISBN 0 582 41923 9 
 
First published 1876                                                                                  
Published by Puffin Books 1950                                                                         
This edition first publis

### Embedding Model(임베딩: 텍스트를 숫자로)

In [16]:
from langchain_openai import OpenAIEmbeddings
import pandas as pd

# 임베딩 모델
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
text="The quick brown for jumps over the lazy dog."
vector = embeddings.embed_query(text) # 하나의 문자열을 벡터로 변환.

print(len(vector))
print(pd.Series(vector).head())

1536
0   -0.003728
1    0.000956
2    0.023028
3   -0.057349
4   -0.011776
dtype: float64


In [18]:
# 문서 내용만 추출
docs = [document.page_content for document in documents]
print(len(docs))

# embed_document() : 문자열 리스트를 받아서, 각가을 벡터로 변환한 뒤 '벡터 리스트'를 반환
vects = embeddings.embed_documents(docs)

print(len(vects))
print(len(vects[0]))

35
35
1536
